# 0.0 Get Raw Data
## Gerardo Leyva Conde

## Library Imports

In [1]:
import os
import re
import json
import base64
import requests
from datetime import datetime
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
def log ( api: str, endpoint: str, data_: str ,status: str, file: str = None) -> None:
    date_today = datetime.now()
    date_formated = date_today.strftime('%Y%m%d')
    log_file = f'{api}_{date_formated}.log'

    log_folder = "data/logs/"

    if not os.path.exists(log_folder):
        os.makedirs(log_folder)

    log_file_path = os.path.join(log_folder, log_file)
    with open(log_file_path, 'a') as file:
        file.write(f"{endpoint}, {date_today}, {status}: {str(data_)}\n")
    return

def save_data ( api_name: str, reference: str, format: str, data: any ) -> None:
    date_today = datetime.now()
    date_formated = date_today.strftime('%Y%m%d')
    raw_folder = 'data/raw/'
    data_file = f'{api_name}_{reference}_{date_formated}.{format}'

    data_path = os.path.join(raw_folder, data_file)

    if isinstance(data, (dict, list)):
        data = json.dumps(data)

    with open( data_path, 'a' ) as file:
        file.write(data)
    return

Get Token Request

In [4]:
os.getenv('CLIENT_ID_SPOTIFY')

'07fee247233647ef88b32f9b25d189f9'

In [5]:


class Spotify:
    def __init__( self ):
        self.__name = 'spotify'
        self.__token = None
        t = os.getenv('SPOTIFY_ENDPOINTS')
        self.__endpoint = json.loads(t)
        self.__getToken()
        pass

    def __getToken ( self ) -> bool:
        url = self.__endpoint['TOKEN']
        try:
            response = requests.post(
                url=url,
                data={
                    'grant_type': 'client_credentials'
                },
                headers={
                    'Authorization': f'Basic {base64.b64encode(f"{os.getenv('CLIENT_ID_SPOTIFY')}:{os.getenv('CLIENT_SECRET_SPOTIFY')}".encode()).decode('utf-8')}'
                }
            )

            data = response.json()
            log(self.__name, url, data, response)
            self.__token = data
            return True
        
        except Exception as error:
            log(self.__name, url, error, "Error")

    def search ( self, type: str, words: str ) -> dict:
        url = self.__endpoint['SEARCH']
        try:
            allowed_types = ["album", "artist", "playlist", "track", "show", "episode", "audiobook"]

            if not type in allowed_types:
                raise Exception(f"Type {type} not valid")
            clean_w = re.sub(r' ', '%20', words)
            params = f"q={clean_w}"
            params += f"&type={type}&limit=10"
            print(params)
            response = requests.get(
                url=url,
                params=params,
                headers={
                    "Authorization": f"{self.__token['token_type']} {self.__token['access_token']}"
                }
            )

            data = response.json()

            log(self.__name, url, data, response)
            save_data(self.__name, 'search', 'json', data)

            return data
        except Exception as error:
            log(self.__name, url, error, "Error")
    
    def topTracks ( self, id_artits ) -> dict:
        url = f'{self.__endpoint['TOPTRACKS']}/{id_artits}/top-tracks'
        try:
            response = requests.get(
                url=url,
                headers={
                    "Authorization": f"{self.__token['token_type']} {self.__token['access_token']}"
                }
            )
            data = response.json()
            log(self.__name, url, data, response)
            save_data(self.__name, 'toptracks', 'json', data)

            return data
        except Exception as error:
            log(self.__name, url, error, "Error")

In [6]:
client = Spotify()

In [7]:
artis_to_search=["Chayanne", "Post Malon", "Radiohead", "Aleks Syntek", "Eros Ramazzotti", "Oasis", "Miranda", "Imagine Dragons", "The Weeknd", "Avenged Sevenfold"]

In [8]:
type='artist'
results = [ client.search(type, artists) for artists in artis_to_search]

q=Chayanne&type=artist&limit=10
q=Post%20Malon&type=artist&limit=10
q=Radiohead&type=artist&limit=10
q=Aleks%20Syntek&type=artist&limit=10
q=Eros%20Ramazzotti&type=artist&limit=10
q=Oasis&type=artist&limit=10
q=Miranda&type=artist&limit=10
q=Imagine%20Dragons&type=artist&limit=10
q=The%20Weeknd&type=artist&limit=10
q=Avenged%20Sevenfold&type=artist&limit=10


In [9]:
print(results)

[{'artists': {'href': 'https://api.spotify.com/v1/search?offset=0&limit=10&query=Chayanne&type=artist', 'limit': 10, 'next': 'https://api.spotify.com/v1/search?offset=10&limit=10&query=Chayanne&type=artist', 'offset': 0, 'previous': None, 'total': 899, 'items': [{'external_urls': {'spotify': 'https://open.spotify.com/artist/1JbemQ1fPt2YmSLjAFhPBv'}, 'followers': {'href': None, 'total': 8680792}, 'genres': ['latin pop'], 'href': 'https://api.spotify.com/v1/artists/1JbemQ1fPt2YmSLjAFhPBv', 'id': '1JbemQ1fPt2YmSLjAFhPBv', 'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5ebcfa5421bfe01fa7eea6308fe', 'height': 640, 'width': 640}, {'url': 'https://i.scdn.co/image/ab67616100005174cfa5421bfe01fa7eea6308fe', 'height': 320, 'width': 320}, {'url': 'https://i.scdn.co/image/ab6761610000f178cfa5421bfe01fa7eea6308fe', 'height': 160, 'width': 160}], 'name': 'Chayanne', 'popularity': 75, 'type': 'artist', 'uri': 'spotify:artist:1JbemQ1fPt2YmSLjAFhPBv'}, {'external_urls': {'spotify': 'https://o

In [10]:
results[0]['artists']['items'][0]

{'external_urls': {'spotify': 'https://open.spotify.com/artist/1JbemQ1fPt2YmSLjAFhPBv'},
 'followers': {'href': None, 'total': 8680792},
 'genres': ['latin pop'],
 'href': 'https://api.spotify.com/v1/artists/1JbemQ1fPt2YmSLjAFhPBv',
 'id': '1JbemQ1fPt2YmSLjAFhPBv',
 'images': [{'url': 'https://i.scdn.co/image/ab6761610000e5ebcfa5421bfe01fa7eea6308fe',
   'height': 640,
   'width': 640},
  {'url': 'https://i.scdn.co/image/ab67616100005174cfa5421bfe01fa7eea6308fe',
   'height': 320,
   'width': 320},
  {'url': 'https://i.scdn.co/image/ab6761610000f178cfa5421bfe01fa7eea6308fe',
   'height': 160,
   'width': 160}],
 'name': 'Chayanne',
 'popularity': 75,
 'type': 'artist',
 'uri': 'spotify:artist:1JbemQ1fPt2YmSLjAFhPBv'}

In [11]:
list_artists = [ {"name": a['name'], "id": a['id']} for a in results[0]['artists']['items']]
list_artists

[{'name': 'Chayanne', 'id': '1JbemQ1fPt2YmSLjAFhPBv'},
 {'name': 'Channel Tres', 'id': '4cUkGQyhLFqKHBtL58HYVp'},
 {'name': 'Luis Miguel', 'id': '2nszmSgqreHSdJA3zWPyrW'},
 {'name': 'Baby Relax Channel', 'id': '1OdSVLNaKmKCNUjJz9sRUh'},
 {'name': 'Ha*Ash', 'id': '5xd2Tg7Zo8755eCy8Gxkp8'},
 {'name': 'Cafe Music BGM channel', 'id': '3yKoHgjtyenCOIRaD2Gghu'},
 {'name': 'Reik', 'id': '0vR2qb8m9WHeZ5ByCbimq2'},
 {'name': 'Chavanté', 'id': '46hfNL2Bni5Ux8hCDMAjIN'},
 {'name': 'Yuridia', 'id': '5B8ApeENp4bE4EE3LI8jK2'},
 {'name': 'BGM channel', 'id': '0ykZILVRZgnmdTi0opgb2f'}]

In [12]:
for e in results:
    list_artists = [ {"name": a['name'], "id": a['id']} for a in e['artists']['items']]
    for a in list_artists:
        top_traks = client.topTracks(a['id'])
        save_data('spotify', f'{a['name']}_toptraks', 'json', top_traks)